
# Splines

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Spline regression models a smooth nonlinear mean function by fitting piecewise polynomials joined at knots.

**Highest-probability exam pattern:** Use splines for nonlinear regression, tune knots or degrees of freedom, and explain flexibility versus overfitting.



## Assumptions, intuition, and theory

Splines are more stable than high-degree global polynomials because each basis function is locally active. More knots increase flexibility and variance. CV is the practical way to choose complexity.



    ## Python method notes and documentation

    `SplineTransformer` creates spline basis columns, `LinearRegression` fits coefficients on those basis columns, and `cross_val_score` estimates prediction MSE for each knot count.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [SplineTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.SplineTransformer.html)
- [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [LeaveOneOut](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for CV results.
import matplotlib.pyplot as plt  # Import plotting tools for smooth curves.
from sklearn.linear_model import LinearRegression  # Import linear regression for the spline basis coefficients.
from sklearn.model_selection import KFold, cross_val_score  # Import CV tools for choosing spline complexity.
from sklearn.pipeline import make_pipeline  # Import pipeline construction.
from sklearn.preprocessing import SplineTransformer  # Import spline basis construction.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_sine_regression_data(n=120, noise=0.35, random_state=123):  # Define a reusable nonlinear regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Draw predictor values and sort them for cleaner plots.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true nonlinear mean function.
    y = signal + rng.normal(0.0, noise, size=n)  # Add Gaussian noise to create observed responses.
    X = x.reshape(-1, 1)  # Convert the predictor to a two-dimensional sklearn design matrix.
    return X, y, signal  # Return predictors, observed response, and noiseless truth.

def make_polynomial_regression_data(n=140, noise=2.0, random_state=123):  # Define a curved polynomial regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = rng.uniform(-2.5, 2.5, size=n)  # Draw one numeric predictor over a moderate range.
    signal = 1.0 - 2.0 * x + 0.8 * x**2 + 0.7 * x**3  # Define the true polynomial mean curve.
    y = signal + rng.normal(0.0, noise, size=n)  # Add independent Gaussian errors.
    X = x.reshape(-1, 1)  # Reshape the predictor for sklearn estimators.
    return X, y, signal  # Return the design matrix, response, and true mean.

def train_test_mse(model, X, y, test_size=0.30, random_state=123):  # Define a local train/test regression evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split observations into training and testing parts.
        X,  # Pass the predictor matrix to the splitter.
        y,  # Pass the numeric response vector to the splitter.
        test_size=test_size,  # Hold out this fraction for honest testing.
        random_state=random_state,  # Fix the random split for reproducibility.
    )  # Finish the train/test split call.
    fitted_model = clone(model)  # Clone the estimator so repeated calls do not reuse fitted state.
    fitted_model.fit(X_train, y_train)  # Fit the model only on the training data.
    train_pred = fitted_model.predict(X_train)  # Predict the training responses to assess in-sample error.
    test_pred = fitted_model.predict(X_test)  # Predict the held-out responses to assess future error.
    train_mse = mean_squared_error(y_train, train_pred)  # Compute training mean squared error.
    test_mse = mean_squared_error(y_test, test_pred)  # Compute test mean squared error.
    return fitted_model, train_mse, test_mse  # Return the fitted model and both error estimates.


In [ ]:
X, y, true_signal = make_sine_regression_data(n=130, noise=0.35, random_state=RANDOM_SEED)  # Simulate nonlinear regression data.
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)  # Define shuffled 5-fold CV.
rows = []  # Create a list for spline CV results.
for n_knots in [3, 4, 5, 6, 8, 10]:  # Try several spline complexities.
    model = make_pipeline(SplineTransformer(n_knots=n_knots, degree=3, include_bias=False), LinearRegression())  # Build cubic spline regression.
    mse_values = -cross_val_score(model, X, y, cv=cv, scoring='neg_mean_squared_error')  # Estimate prediction MSE by CV.
    rows.append({'n_knots': n_knots, 'cv_mse': np.mean(mse_values)})  # Store average CV MSE.
results = pd.DataFrame(rows)  # Convert results to a table.
best_knots = int(results.loc[results['cv_mse'].idxmin(), 'n_knots'])  # Select the knot count with smallest CV MSE.
best_model = make_pipeline(SplineTransformer(n_knots=best_knots, degree=3, include_bias=False), LinearRegression())  # Rebuild the best spline model.
best_model.fit(X, y)  # Fit the selected spline model on the full data for plotting.
x_grid = np.linspace(X[:, 0].min(), X[:, 0].max(), 250).reshape(-1, 1)  # Create a dense grid for the smooth fitted curve.
y_grid = best_model.predict(x_grid)  # Predict the spline curve on the grid.
display(results)  # Display the knot-selection table.
plt.figure(figsize=(6, 4))  # Create a figure for the spline fit.
plt.scatter(X[:, 0], y, s=22, alpha=0.6, label='observed data')  # Plot observed points.
plt.plot(x_grid[:, 0], y_grid, color='red', label=f'{best_knots} knots')  # Plot the selected spline curve.
plt.xlabel('x')  # Label the predictor axis.
plt.ylabel('y')  # Label the response axis.
plt.title('Spline regression selected by CV')  # Title the fit plot.
plt.legend()  # Show labels.
plt.show()  # Render the fit plot.
plt.figure(figsize=(6, 4))  # Create a figure for CV error.
plt.plot(results['n_knots'], results['cv_mse'], marker='o')  # Plot CV MSE against knot count.
plt.xlabel('number of knots')  # Label the tuning axis.
plt.ylabel('CV MSE')  # Label the error axis.
plt.title('Spline complexity tuning')  # Title the tuning plot.
plt.show()  # Render the CV plot.



## Exam-style problem prompt

A nonlinear regression prompt asks for a smoother than a straight line but less wild than a high-degree polynomial. Fit spline models with several knot counts and choose by CV MSE.



    ## Adaptable code pattern

    ```python
    # Step 1: Create candidate numbers of knots or degrees of freedom.
# Step 2: For each candidate, build SplineTransformer + LinearRegression.
# Step 3: Estimate CV MSE.
# Step 4: Choose the smallest CV MSE.
# Step 5: Plot observed data and the selected smooth curve.
# Step 6: Interpret knots as a flexibility control.
    ```



## What to remember for the exam

A spline is just linear regression on spline basis functions. On an exam, emphasize knot choice, smoothness, and CV error.
